# 04_Data_preprocessing.ipynb

## Giới thiệu
Notebook này được xây dựng với mục tiêu **chuẩn hóa dữ liệu** trước khi huấn luyện mô hình.  
Các bước chính bao gồm:

- **Mã hóa dữ liệu**: áp dụng các kỹ thuật như One-Hot Encoding, Label Encoding, Ordinal Encoding để biến đổi biến phân loại thành dạng số.  
- **Chuẩn hóa dữ liệu số**: sử dụng Min-Max Scaler hoặc Standard Scaler để đưa dữ liệu về cùng thang đo.  
- **Tạo file dữ liệu cuối cùng (`final`)**: sau khi xử lý, dữ liệu được lưu lại để sử dụng cho quá trình huấn luyện mô hình.  
- **Chia tập train/test**: tách dữ liệu thành tập huấn luyện và tập kiểm thử, đảm bảo quy trình đánh giá mô hình khách quan.  

## Vai trò
Notebook này đóng vai trò là **bước trung gian quan trọng** trong pipeline, giúp dữ liệu từ trạng thái thô (raw/clean/features) trở thành **dữ liệu sẵn sàng cho machine learning**.


In [1]:
import pandas as pd
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    OneHotEncoder,
    LabelEncoder,
    OrdinalEncoder,
)
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [2]:
data_path = r'D:\Projects\Machine_learning\IEEE-CIS-Fraud-Detection\Data\03_Features\data_features_v2.csv'
df = pd.read_csv(data_path)
print('Đọc file data thành công')

Đọc file data thành công


C:\Users\LỘC2007\AppData\Local\Temp\ipykernel_12772\513048385.py:2: DtypeWarning: Columns (0: CNT_FAM_MEMBERS_GROUP) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path)


### CHỌN PHÂN CÁC CỘT CHO TỪNG LOẠI PHƯƠNG PHÁP MÃ HÓA

In [3]:
numeric_cols_tem = df.select_dtypes(include= 'number') 
cateory_cols = df.select_dtypes(include= 'string') 

In [4]:
booled_cols = []
number_cols = []
for col in numeric_cols_tem: 
    if len(df[col].unique()) == 2:  # giá trị của các cột chỉ có giá trị 0 và 1
        booled_cols.append(col)
    else:
        number_cols.append(col)
print(booled_cols)
print(number_cols)

['TARGET', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_7', 'FLAG_DOCUMENT_8', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_11', 'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16', 'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_21', 'FLAG_OVER_BORROWING']
['CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'EXT_SOURCE_1', 'EXT_SOURCE_2'

### Phân loại theo độ chênh lệch giữa mean và median để đưa ra quyết định cho các cột có giá trị là số 

In [5]:
# In header
print(f"{'Column'.ljust(25)}{'Mean'.rjust(15)}{'Median'.rjust(15)}{'Diff %'.rjust(15)}")
print("-"*70)

# In từng dòng
for col in number_cols:
    mean_val = df[col].mean()
    median_val = df[col].median()
    diff_percent = abs(((mean_val - median_val) / median_val * 100)) if median_val != 0 else 0
    
    print(f"{col.ljust(25)}{str(round(mean_val,2)).rjust(15)}{str(round(median_val,2)).rjust(15)}{str(round(diff_percent,2)).rjust(15)}")


Column                              Mean         Median         Diff %
----------------------------------------------------------------------
CNT_CHILDREN                         0.5            0.0              0
AMT_INCOME_TOTAL                176735.9       157500.0          12.21
AMT_CREDIT                     613413.03       521280.0          17.67
AMT_ANNUITY                      27837.1        25803.0           7.88
AMT_GOODS_PRICE                551246.41       450000.0           22.5
REGION_POPULATION_RELATIVE           0.02           0.02          10.03
DAYS_BIRTH                     -14904.56       -14356.0           3.82
DAYS_EMPLOYED                   35780.76        -1354.0         2742.6
DAYS_REGISTRATION               -4521.92        -4132.0           9.44
DAYS_ID_PUBLISH                 -2937.04        -3107.0           5.47
OWN_CAR_AGE                          5.0            1.0         400.38
FLAG_MOBIL                           1.0            1.0            0.0
CNT_F

In [6]:
df

,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,INCOME_PER_PERSON,GOODS_TO_CREDIT_RATIO,LTV,DOWN_PAYMENT,FLAG_OVER_BORROWING,EMPLOYED_TO_AGE,CAR_TO_AGE,EXT_SOURCES_PROD,EXT_SOURCES_MEAN,EXT_SOURCES_STD
0,1,Cash loans,M,N,Y,0,202500.000,406597.5,24700.5,351000.0,...,202497.975020,0.863262,1.158397,-55597.5,1,0.067329,0.038579,0.003043,0.161787,0.092026
1,0,Cash loans,F,Y,Y,1,171000.000,1560726.0,41301.0,1395000.0,...,56999.810001,0.893815,1.118800,-165726.0,1,0.227174,0.450356,0.276010,0.663607,0.150717
2,0,Cash loans,F,N,Y,0,112500.000,1019610.0,33826.5,913500.0,...,56249.718751,0.895931,1.116158,-106110.0,1,18.172198,0.018160,0.090840,0.514935,0.280096
3,0,Cash loans,F,N,Y,1,112500.000,652500.0,21177.0,652500.0,...,37499.875000,1.000000,1.000000,0.0,0,0.066588,0.035795,0.075861,0.445189,0.180342
4,0,Cash loans,F,N,Y,0,38419.155,148365.0,10678.5,135000.0,...,19209.481453,0.909918,1.099000,-13365.0,1,17.889161,0.017877,0.261725,0.643375,0.083837
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
126390,0,Cash loans,F,N,Y,0,261000.000,1303812.0,35982.0,1138500.0,...,130499.347503,0.873209,1.145202,-165312.0,1,0.261206,0.017901,0.238845,0.674368,0.296423
126391,0,Cash loans,F,N,Y,0,112500.000,345510.0,17770.5,247500.0,...,112498.875011,0.716332,1.396000,-98010.0,1,0.033614,0.030750,0.074350,0.451321,0.187941
126392,0,Cash loans,F,N,Y,0,153000.000,677664.0,29979.0,585000.0,...,152998.470015,0.863260,1.158400,-92664.0,1,0.529266,0.024389,0.087235,0.499536,0.264447
126393,1,Cash loans,F,N,Y,0,171000.000,370107.0,20205.0,319500.0,...,85499.572502,0.863264,1.158394,-50607.0,1,0.400134,0.030516,0.171794,0.560217,0.087410


# Giải Thích Chiến Lược Chuẩn Hóa Dữ Liệu (Scaling Strategy)

Để tối ưu hóa hiệu năng cho các mô hình học máy nhạy cảm với khoảng giá trị (hoặc chuẩn bị sẵn sàng cho việc thử nghiệm nhiều thuật toán khác nhau như Logistic Regression, SVM, Neural Networks), chúng ta chia các cột số thành 3 nhóm chuẩn hóa chuyên biệt dựa trên **Business Logic** và **Độ lệch phân phối (Diff % giữa Mean & Median)**:

---

### Nhóm 1: MinMaxScaler — Các biến số có giới hạn cứng (Bounded) & Cân bằng
* **Danh sách cột:** `EXT_SOURCE_1`, `EXT_SOURCE_2`, `EXT_SOURCE_3`
* **Cơ sở phân tích:** 
  * Độ lệch giữa Mean và Median cực kỳ thấp (lần lượt là $0\%$, $9.12\%$, và $1.22\%$). Điều này chứng tỏ dữ liệu phân phối rất đều và đối xứng.
  * Bản chất các chỉ số `EXT_SOURCE` là điểm số tín dụng từ các tổ chức thứ ba, đã được chuẩn hóa sẵn trong khoảng $[0, 1]$.
* **Lý do dùng MinMaxScaler:** Khi dữ liệu đã có biên giới hạn cứng rõ ràng và hoàn toàn không bị nhiễu bởi Outliers (giá trị ngoại lai), MinMaxScaler giúp co giãn dữ liệu về đúng khoảng $[0, 1]$ mà **không làm thay đổi hình dáng phân phối gốc** của điểm số.

---

### Nhóm 2: RobustScaler — Các biến số lệch nặng & Nhiều Outliers đột biến
* **Danh sách cột:** `DAYS_EMPLOYED`, `EMPLOYED_TO_AGE`, `OWN_CAR_AGE`, `CAR_TO_AGE`, `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `AMT_GOODS_PRICE`, `INCOME_PER_PERSON`, `DOWN_PAYMENT`, `AMT_REQ_CREDIT_BUREAU_YEAR`, `EXT_SOURCES_PROD`
* **Cơ sở phân tích:**
  * Các cột này có độ lệch `Diff %` rất lớn (từ $12\%$ đến hơn $2700\%$).
  * `DAYS_EMPLOYED` chứa giá trị rác hệ thống là `365243` (mã hóa người nghỉ hưu) kéo lệch toàn bộ dữ liệu.
  * Các cột tài chính (`AMT_INCOME_TOTAL`, `AMT_CREDIT`) bị lệch nặng do phân phối đuôi dài (luôn có nhóm cực giàu vay khoản cực lớn).
  * `OWN_CAR_AGE` bị lệch do đa số xe rất mới (Median = 1) nhưng có một số xe cổ rất cũ kéo Mean lên 5 năm.
* **Lý do dùng RobustScaler:** MinMaxScaler hay StandardScaler sẽ bị hỏng hoàn toàn trước các giá trị ngoại lai khổng lồ này (khiến $99\%$ dữ liệu bình thường bị ép chặt vào một khoảng siêu nhỏ). RobustScaler sử dụng **Trung vị (Median)** và **Khoảng tứ phân vị (IQR)** để scale, giúp gạt bỏ hoàn toàn ảnh hưởng của nhiễu và giữ nguyên độ biến thiên của số đông.

---

### Nhóm 3: StandardScaler — Các biến số phân phối tự nhiên, ít Outliers
* **Danh sách cột:** `CNT_CHILDREN`, `AMT_ANNUITY`, `REGION_POPULATION_RELATIVE`, `DAYS_BIRTH` (Tuổi), `DAYS_REGISTRATION`, `DAYS_ID_PUBLISH`, `CNT_FAM_MEMBERS`, `EXT_SOURCES_MEAN`, `EXT_SOURCES_STD`, `DAYS_LAST_PHONE_CHANGE`
* **Cơ sở phân tích:**
  * Độ lệch `Diff %` ở mức thấp hoặc trung bình.
  * Các cột như tuổi tác (`DAYS_BIRTH`), số con (`CNT_CHILDREN`), số thành viên gia đình (`CNT_FAM_MEMBERS`) tăng giảm tuyến tính theo tự nhiên, có giới hạn thực tế nên không thể xuất hiện các Outliers dị biệt (không ai có 100 con hay 200 tuổi).
  * Khoản trả góp hàng tháng (`AMT_ANNUITY`) dù là biến tài chính nhưng bị giới hạn bởi trần duyệt vay của ngân hàng nên không bị vỡ phân phối.
* **Lý do dùng StandardScaler:** Đưa dữ liệu về dạng chuẩn có Trung bình ($\mu = 0$) và Độ lệch chuẩn ($\sigma = 1$). Đây là trạng thái toán học lý tưởng nhất cho đa số thuật toán, giúp mô hình hội tụ nhanh và ổn định.

In [7]:
df['EMPLOYED_TO_AGE'] = abs(df['EMPLOYED_TO_AGE'])

In [8]:
# Nếu chọn dữ liệu gốc không gọp nhóm 
# 1. Làn MinMax: Cho các cột điểm số/tỷ lệ đã giới hạn sẵn [0, 1] và cực kỳ cân bằng
minmax_cols = [
    'EXT_SOURCE_1', 
    'EXT_SOURCE_2', 
    'EXT_SOURCE_3',

]

# 2. Làn Robust: Cho các cột có độ lệch lớn (Diff % > 12%), chứa nhiều giá trị ngoại lai
# (Thêm CNT_FAM_MEMBERS vì Diff % = 12.4%, DAYS_LAST_PHONE_CHANGE vì Diff % = 21.99%)
skewed_cols = [
    
    'AMT_INCOME_TOTAL', # Thu nhập gốc vẫn giữ lại làm biến liên tục vì NAME_INCOME_GROUP của bạn là gộp loại hình thu nhập (Working, Pensioner,...)
    'AMT_CREDIT', 
    'AMT_ANNUITY', 
    'AMT_GOODS_PRICE',
    'OWN_CAR_AGE',
    # Các biến truy vấn tín dụng ngắn hạn (giữ nguyên vì chưa gộp)
    'AMT_REQ_CREDIT_BUREAU_HOUR', 
    'AMT_REQ_CREDIT_BUREAU_DAY', 
    'AMT_REQ_CREDIT_BUREAU_WEEK', 
    'AMT_REQ_CREDIT_BUREAU_MON', 
    'AMT_REQ_CREDIT_BUREAU_QRT', 
    # Các chỉ số tỉ lệ (Ratios) tự tạo
    'ANNUITY_TO_INCOME_RATIO', 
    'CREDIT_TO_INCOME', 
    'PAYMENT_RATE', 
    'INCOME_PER_PERSON', 
    'LTV', 
    'DOWN_PAYMENT',
    'CAR_TO_AGE'
]

# 3. Làn Standard: Cho các cột số phân phối đều, ít nhạy cảm với outliers (Diff % < 12%)
normal_num_cols = [
    'AGE',
    'DAYS_EMPLOYED', 
    'DAYS_REGISTRATION', 
    'DAYS_ID_PUBLISH', 
    'DAYS_LAST_PHONE_CHANGE',
    'EXT_SOURCES_STD',
    'EXT_SOURCES_PROD', 
    'EXT_SOURCES_MEAN',
    'REGION_POPULATION_RELATIVE',
    'GOODS_TO_CREDIT_RATIO',
    'EMPLOYED_TO_AGE'
]

In [9]:
# In header
print(f"{'Column'.ljust(25)}{'Count unique values'.rjust(25)}")
print("-"*70)

# In từng dòng
for col in cateory_cols:
    len_unique = len(df[col].unique())    
    print(f"{col.ljust(25)}{str(round(len_unique,2)).rjust(25)}")

for col in cateory_cols:
    print('\n\n',col)
    print(f'{df[col].unique()}')
    print('=='*20)


Column                         Count unique values
----------------------------------------------------------------------
NAME_CONTRACT_TYPE                               2
CODE_GENDER                                      3
FLAG_OWN_CAR                                     2
FLAG_OWN_REALTY                                  2
NAME_INCOME_TYPE                                 8
NAME_EDUCATION_TYPE                              5
NAME_FAMILY_STATUS                               5
NAME_HOUSING_TYPE                                6
OCCUPATION_TYPE                                 18
ORGANIZATION_TYPE                               58
CNT_CHILDREN_GROUP                               5
NAME_EDUCATION_GROUP                             3
ORGANIZATION_GROUP                               9
NAME_INCOME_GROUP                                5
NAME_HOUSING_GROUP                               3
DEF_30_GROUP                                     4
DEF_60_GROUP                                     4
OBS_30_GROU

In [10]:
ordinal_col = [
    'NAME_EDUCATION_GROUP',      # Gộp từ NAME_EDUCATION_TYPE
    'CNT_CHILDREN_GROUP',        # Gộp từ CNT_CHILDREN
    'CNT_FAM_MEMBERS_GROUP',     # Gộp từ CNT_FAM_MEMBERS
    'DEF_30_GROUP',              # Gộp từ DEF_30_CNT_SOCIAL_CIRCLE
    'DEF_60_GROUP',              # Gộp từ DEF_60_CNT_SOCIAL_CIRCLE (theo dòng dữ liệu mẫu)
    'OBS_30_GROUP',              # Gộp từ OBS_30_CNT_SOCIAL_CIRCLE (theo dòng dữ liệu mẫu)
    'OBS_60_GROUP',              # Gộp từ OBS_60_CNT_SOCIAL_CIRCLE (theo dòng dữ liệu mẫu)
    'BUREAU_YEAR_GROUP',         # Gộp từ AMT_REQ_CREDIT_BUREAU_YEAR
    'YEARS_BEGIN_GROUP',         # Gộp từ YEARS_BEGINEXPLUATATION_AVG
    'REGION_RATING_CLIENT', 
    'REGION_RATING_CLIENT_W_CITY'
]

In [11]:
onehot_col = [
    'NAME_CONTRACT_TYPE', 
    'CODE_GENDER', 
    'FLAG_OWN_CAR', 
    'FLAG_OWN_REALTY', 
    'NAME_FAMILY_STATUS', 
    'ORGANIZATION_GROUP',        # Gộp từ ORGANIZATION_TYPE
    'NAME_INCOME_GROUP',         # Gộp từ NAME_INCOME_TYPE
    'NAME_HOUSING_GROUP'         # Gộp từ NAME_HOUSING_TYPE
]

In [ ]:
binary_cols = [
    "TARGET",
    "FLAG_MOBIL",
    "FLAG_EMP_PHONE",
    "FLAG_WORK_PHONE",
    "FLAG_CONT_MOBILE",
    "FLAG_PHONE",
    "FLAG_EMAIL",
    "REG_REGION_NOT_LIVE_REGION",
    "REG_REGION_NOT_WORK_REGION",
    "LIVE_REGION_NOT_WORK_REGION",
    "REG_CITY_NOT_LIVE_CITY",
    "REG_CITY_NOT_WORK_CITY",
    "LIVE_CITY_NOT_WORK_CITY",
    "FLAG_DOCUMENT_2",
    "FLAG_DOCUMENT_3",
    "FLAG_DOCUMENT_4",
    "FLAG_DOCUMENT_5",
    "FLAG_DOCUMENT_6",
    "FLAG_DOCUMENT_7",
    "FLAG_DOCUMENT_8",
    "FLAG_DOCUMENT_9",
    "FLAG_DOCUMENT_10",
    "FLAG_DOCUMENT_11",
    "FLAG_DOCUMENT_12",
    "FLAG_DOCUMENT_13",
    "FLAG_DOCUMENT_14",
    "FLAG_DOCUMENT_15",
    "FLAG_DOCUMENT_16",
    "FLAG_DOCUMENT_17",
    "FLAG_DOCUMENT_18",
    "FLAG_DOCUMENT_19",
    "FLAG_DOCUMENT_20",
    "FLAG_DOCUMENT_21",
    "FLAG_OVER_BORROWING"
]


In [12]:
label_encoder_cols = [
    'OCCUPATION_TYPE'            # Giữ lại vì loại công việc chi tiết vẫn rất giá trị
]

In [13]:
# Thiết lập các nhóm thứ tự tăng dần từ trái qua phải
education_order = ['Basic_Education', 'Incomplete higher', 'Advanced_Education']
children_order = ['0', '1', '2', '3', '>3']
fam_members_order = ['1', '2', '3', '4', '5', '6', '>6']
def_30_obs_order = ['0', '1', '2-4', '>=5']
def_60_order = ['0', '1', '2', '>=3']  # Đã sửa theo đúng output thực tế của bạn
bureau_order = ['0', '1', '2-3', '4-6', '>=7']
region_order = [1, 2, 3]  # Giữ nguyên kiểu int gốc của rating vùng
years_begin_order = ['1','2','3','4','>=5']
# Đảm bảo thứ tự các list này khớp 100% với thứ tự khai báo trong danh sách ordinal_col
ordinal_categories = [
    education_order,     # NAME_EDUCATION_GROUP
    children_order,      # CNT_CHILDREN_GROUP
    fam_members_order,   # CNT_FAM_MEMBERS_GROUP
    def_30_obs_order,    # DEF_30_GROUP
    def_60_order,        # DEF_60_GROUP
    def_30_obs_order,    # OBS_30_GROUP
    def_30_obs_order,    # OBS_60_GROUP
    bureau_order,        # BUREAU_YEAR_GROUP
    years_begin_order,   # YEARS_BEGIN_GROUP
    region_order,        # REGION_RATING_CLIENT
    region_order         # REGION_RATING_CLIENT_W_CITY
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        # 1. Nhóm lệch nặng (Sử dụng RobustScaler)
        ('num_skewed', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),  
            ('scaler', RobustScaler())
        ]), skewed_cols),

        # 2. Nhóm phân phối chuẩn (Sử dụng StandardScaler)
        ('num_normal', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ]), normal_num_cols),
        
        # 3. Nhóm điểm số giới hạn [0, 1] (Sử dụng MinMaxScaler)
        ('min_max', Pipeline([
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', MinMaxScaler())
        ]), minmax_cols),

        # 4. Mã hóa ordinal (Áp dụng chính xác thứ tự logic đã khai báo)
        ('categorical_ordinal', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(
                categories=ordinal_categories, 
                handle_unknown='use_encoded_value', 
                unknown_value=-1
            ))
        ]), ordinal_col),

        # 5. Mã hóa onehot (Dành cho các biến phân loại không thứ tự, ít nhãn)
        ('categorical_onehot', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), onehot_col),

        # 6. Mã hóa các biến phân loại nhiều nhãn (OCCUPATION_TYPE)
        ('categorical_high_card', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
        ]), label_encoder_cols),

        # 7. Nhóm các cột nhị phân (Binary Flags 0/1) - Giữ nguyên gốc không scale
        # ('binary_flags', 'passthrough', binary_cols)
        binary_cols
    ],
    remainder='drop'
)

In [15]:
# Ép đầu ra thành Pandas DataFrame
preprocessor.set_output(transform="pandas")

# TRUYỀN THẲNG TOÀN BỘ DF gốc vào đây
# Pipeline sẽ tự nhặt đúng những cột cần thiết và tự động loại bỏ các cột thừa khác
df_processed = preprocessor.fit_transform(df)

In [16]:
df_processed.info()

<class 'pandas.DataFrame'>
RangeIndex: 126395 entries, 0 to 126394
Data columns (total 73 columns):
 #   Column                                                       Non-Null Count   Dtype  
---  ------                                                       --------------   -----  
 0   num_skewed__AMT_INCOME_TOTAL                                 126395 non-null  float64
 1   num_skewed__AMT_CREDIT                                       126395 non-null  float64
 2   num_skewed__AMT_ANNUITY                                      126395 non-null  float64
 3   num_skewed__AMT_GOODS_PRICE                                  126395 non-null  float64
 4   num_skewed__OWN_CAR_AGE                                      126395 non-null  float64
 5   num_skewed__AMT_REQ_CREDIT_BUREAU_HOUR                       126395 non-null  float64
 6   num_skewed__AMT_REQ_CREDIT_BUREAU_DAY                        126395 non-null  float64
 7   num_skewed__AMT_REQ_CREDIT_BUREAU_WEEK                       126395 non-null

In [17]:
import pandas as pd

# Tính ma trận tương quan
corr_matrix = df_processed.corr()

# Lấy các cặp có tương quan >= 0.2
high_corr = corr_matrix[(corr_matrix >= 0.2) | (corr_matrix <= -0.2)]

# In ra danh sách các cột có ít nhất một tương quan cao
cols_high_corr = high_corr.columns[(high_corr.abs().sum() > 1)]  # >1 vì mỗi cột luôn có tương quan 1 với chính nó
print("Các cột có tương quan >= 0.2:", list(cols_high_corr))


Các cột có tương quan >= 0.2: ['num_skewed__AMT_INCOME_TOTAL', 'num_skewed__AMT_CREDIT', 'num_skewed__AMT_ANNUITY', 'num_skewed__AMT_GOODS_PRICE', 'num_skewed__OWN_CAR_AGE', 'num_skewed__AMT_REQ_CREDIT_BUREAU_HOUR', 'num_skewed__AMT_REQ_CREDIT_BUREAU_DAY', 'num_skewed__AMT_REQ_CREDIT_BUREAU_WEEK', 'num_skewed__ANNUITY_TO_INCOME_RATIO', 'num_skewed__CREDIT_TO_INCOME', 'num_skewed__PAYMENT_RATE', 'num_skewed__INCOME_PER_PERSON', 'num_skewed__LTV', 'num_skewed__DOWN_PAYMENT', 'num_skewed__CAR_TO_AGE', 'num_normal__AGE', 'num_normal__DAYS_EMPLOYED', 'num_normal__DAYS_REGISTRATION', 'num_normal__DAYS_LAST_PHONE_CHANGE', 'num_normal__EXT_SOURCES_STD', 'num_normal__EXT_SOURCES_PROD', 'num_normal__EXT_SOURCES_MEAN', 'num_normal__REGION_POPULATION_RELATIVE', 'num_normal__GOODS_TO_CREDIT_RATIO', 'num_normal__EMPLOYED_TO_AGE', 'min_max__EXT_SOURCE_1', 'min_max__EXT_SOURCE_2', 'min_max__EXT_SOURCE_3', 'categorical_ordinal__CNT_CHILDREN_GROUP', 'categorical_ordinal__CNT_FAM_MEMBERS_GROUP', 'categor

In [19]:
df_processed.to_csv('encoder.csv',index=False)

In [33]:
cols_df = df.columns
for col in cols_df:
    if col in df_processed.columns:
        pass
    else:
        if len(df[col].unique()) <= 2:
            if df[col].unique()[0] in ["1","0",1,0] and df[col].unique()[0] in ["1","0",1,0]:
                print(col)
                print(df[col].unique())


TARGET
[1 0]
FLAG_MOBIL
[1]
FLAG_EMP_PHONE
[1 0]
FLAG_WORK_PHONE
[0 1]
FLAG_CONT_MOBILE
[1 0]
FLAG_PHONE
[1 0]
FLAG_EMAIL
[0 1]
REG_REGION_NOT_LIVE_REGION
[0 1]
REG_REGION_NOT_WORK_REGION
[0 1]
LIVE_REGION_NOT_WORK_REGION
[0 1]
REG_CITY_NOT_LIVE_CITY
[0 1]
REG_CITY_NOT_WORK_CITY
[0 1]
LIVE_CITY_NOT_WORK_CITY
[0 1]
FLAG_DOCUMENT_2
[0 1]
FLAG_DOCUMENT_3
[1 0]
FLAG_DOCUMENT_4
[0 1]
FLAG_DOCUMENT_5
[0 1]
FLAG_DOCUMENT_6
[0 1]
FLAG_DOCUMENT_7
[0 1]
FLAG_DOCUMENT_8
[0 1]
FLAG_DOCUMENT_9
[0 1]
FLAG_DOCUMENT_10
[0 1]
FLAG_DOCUMENT_11
[0 1]
FLAG_DOCUMENT_12
[0 1]
FLAG_DOCUMENT_13
[0 1]
FLAG_DOCUMENT_14
[0 1]
FLAG_DOCUMENT_15
[0 1]
FLAG_DOCUMENT_16
[0 1]
FLAG_DOCUMENT_17
[0 1]
FLAG_DOCUMENT_18
[0 1]
FLAG_DOCUMENT_19
[0 1]
FLAG_DOCUMENT_20
[0 1]
FLAG_DOCUMENT_21
[0 1]
FLAG_OVER_BORROWING
[1 0]


In [29]:
df['FLAG_MOBIL']

0         1
1         1
2         1
3         1
4         1
         ..
126390    1
126391    1
126392    1
126393    1
126394    1
Name: FLAG_MOBIL, Length: 126395, dtype: int64